# DeFiAgent-X Lite — provenance-clean Week-4 gate

Run this notebook top to bottom in a CUDA-enabled Google Colab runtime. Add only
`FORK_RPC_URL` to Colab Secrets and enable notebook access. The notebook clones
the public repository, checks out the exact Phase-A source commit, verifies all
pins, launches Anvil and runs the gate in one cell, and downloads the complete
run archive.

Do not add GitHub tokens, wallet keys, or model tokens to the notebook.


In [ ]:
# Fail before installation or model download if a GPU runtime was not selected.
import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then reconnect."
print("CUDA device:", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import userdata

FORK_RPC_URL = userdata.get("FORK_RPC_URL")
assert FORK_RPC_URL and FORK_RPC_URL.startswith("https://"), "Add FORK_RPC_URL to Colab Secrets."
print("Archive endpoint loaded from Colab Secrets (value intentionally not printed).")


In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/esseya22-cpu/defiagent-x-lite.git"
REPO = Path("/content/defiagent-x-lite")

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "submodule", "update", "--init", "--recursive"], cwd=REPO, check=True)

actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Cloned clean source at commit:", actual_commit)


In [ ]:
import os

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"

os.environ.update(
    {
        "FORK_RPC_URL": FORK_RPC_URL,
        "FORK_BLOCK": "20000000",
        "LOCAL_RPC_URL": "http://127.0.0.1:8545",
        # Publicly documented Foundry test mnemonic; funds only disposable fork accounts.
        "ANVIL_MNEMONIC": "test test test test test test test test test test test junk",
        "SETUP_TIME_OFFSET_SECONDS": "1000",
        "DECISION_TIME_OFFSET_SECONDS": "2000",
        "PRIMARY_MODEL_ID": MODEL_ID,
        "PRIMARY_MODEL_REVISION": MODEL_REVISION,
        "ALLOW_BROADCAST": "0",
        "TOKENIZERS_PARALLELISM": "false",
    }
)
print("Experiment environment set; FORK_RPC_URL remains redacted.")


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Install the exact tool versions used by the repository, then honor uv.lock.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.12.7"], check=True)

foundry_bin = Path.home() / ".foundry" / "bin"
os.environ["PATH"] = f"{foundry_bin}:{os.environ['PATH']}"
if shutil.which("foundryup") is None:
    subprocess.run(
        ["bash", "-lc", "curl -L https://foundry.paradigm.xyz | bash"],
        check=True,
    )
subprocess.run([str(foundry_bin / "foundryup"), "--install", "v1.8.1"], check=True)

subprocess.run(["uv", "sync", "--locked", "--all-extras", "--dev"], cwd=REPO, check=True)
subprocess.run(["forge", "build"], cwd=REPO, check=True)
subprocess.run(["uv", "run", "defiagent-week4", "export-schema"], cwd=REPO, check=True)

expected_submodules = {
    "lib/forge-std": "bf647bd6046f2f7da30d0c2bf435e5c76a780c1b",
    "lib/openzeppelin-contracts": "c64a1edb67b6e3f4a15cca8909c9482ad33a02b0",
}
for path, expected in expected_submodules.items():
    actual = subprocess.check_output(["git", "-C", path, "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
    assert actual == expected, f"{path}: expected {expected}, got {actual}"

print(subprocess.check_output(["uv", "--version"], text=True).strip())
print(subprocess.check_output(["forge", "--version"], text=True).splitlines()[0])
print("Pinned submodules verified.")


In [ ]:
# Enforce the immutable model revision before the evidence run.
import subprocess

probe = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        (
            "from huggingface_hub import HfApi; "
            f"i=HfApi().model_info('{MODEL_ID}', revision='{MODEL_REVISION}'); "
            f"assert i.sha == '{MODEL_REVISION}', (i.sha, '{MODEL_REVISION}'); "
            "print(i.sha)"
        ),
    ],
    cwd=REPO,
    check=True,
    text=True,
    capture_output=True,
)
assert probe.stdout.strip() == MODEL_REVISION
assert os.environ["PRIMARY_MODEL_REVISION"] == MODEL_REVISION
assert os.environ["PRIMARY_MODEL_REVISION"] != "main"
print("Immutable model revision verified:", probe.stdout.strip())


## Gate cell

This single cell restarts Anvil, waits for readiness, runs diagnostics, static
quality checks, fork tests, and the complete 20-repetition Qwen gate, validates
the report, creates the archive, and then terminates Anvil in `finally`.


In [ ]:
import json
import os
import signal
import subprocess
import tarfile
import time
from pathlib import Path

from google.colab import files

env = os.environ.copy()
anvil_log_path = Path("/content/anvil-week4.log")

# Restart any disposable process bound to the local research RPC port.
subprocess.run(
    ["bash", "-lc", "command -v fuser >/dev/null && fuser -k 8545/tcp >/dev/null 2>&1 || true"],
    check=True,
)

anvil_log = anvil_log_path.open("w", encoding="utf-8")
anvil = subprocess.Popen(
        ["bash", "scripts/start_anvil.sh"],
"--fork-block-number",
        env["FORK_BLOCK"],
        "--mnemonic",
        env["ANVIL_MNEMONIC"],
        "--host",
        "127.0.0.1",
        "--port",
        "8545",
        "--silent",
    ],
    cwd=REPO,
    env=env,
    stdout=anvil_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

try:
    for _ in range(120):
        if anvil.poll() is not None:
            raise RuntimeError(f"Anvil exited early; inspect {anvil_log_path}")
        ready = subprocess.run(
            ["cast", "block-number", "--rpc-url", env["LOCAL_RPC_URL"]],
            cwd=REPO,
            env=env,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        if ready.returncode == 0:
            break
        time.sleep(1)
    else:
        raise TimeoutError(f"Anvil did not become ready; inspect {anvil_log_path}")

    commands = [
        ["uv", "run", "defiagent-week4", "doctor"],
        ["uv", "run", "ruff", "check", "src", "tests"],
        ["uv", "run", "mypy", "src"],
        ["uv", "run", "pytest"],
        ["forge", "fmt", "--check"],
        ["forge", "test", "--match-contract", "Week4ForkGateTest", "-vv"],
        ["uv", "run", "defiagent-week4", "gate", "--planner", "qwen", "--repetitions", "20"],
    ]
    for command in commands:
        print("+", " ".join(command), flush=True)
        subprocess.run(command, cwd=REPO, env=env, check=True)

    pointer = json.loads((REPO / "results/latest-week4-run.json").read_text())
    run_id = pointer["experiment_id"]
    run_dir = REPO / "results/week4" / run_id
    report = json.loads((run_dir / "week4-gate-report.json").read_text())
    records = [
        json.loads(line)
        for line in (run_dir / "raw/week4-runs.jsonl").read_text().splitlines()
        if line.strip()
    ]
    assert report["experiment_id"] == run_id
    assert report["passed"] is True, report
    assert len(report["checks"]) == 7 and all(report["checks"].values()), report["checks"]
    assert records and {row["model_revision"] for row in records} == {MODEL_REVISION}

    archive = Path("/content") / f"{run_id}.tar.gz"
    with tarfile.open(archive, "w:gz") as tf:
        tf.add(run_dir, arcname=run_id)

    print(json.dumps(report, indent=2))
    print("Run directory:", run_dir)
    print("Archive:", archive)
    files.download(str(archive))
finally:
    if anvil.poll() is None:
        os.killpg(anvil.pid, signal.SIGTERM)
        try:
            anvil.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(anvil.pid, signal.SIGKILL)
            anvil.wait(timeout=5)
    anvil_log.close()
